# Ejecución del Pipeline · Bloque 1 (Audio)

Notebook para ejecutar y verificar cada etapa del pipeline de audio paso a paso.

**Orden de ejecución:**
1. Verificar prerrequisitos
2. Descargar vídeo y transcripción oficial
3. Extraer audio (WAV)
4. Transcripción (ASR)
5. Diarización de hablantes
6. Cruce transcript + diarización → `transcript.csv`
7. Emoción de voz → `voice.csv`
8. Síntesis de voz (TTS) → `briefing.mp3` *(solo si B2 generó `summary.json`)*

## 0. Prerrequisitos

Verificar que todo está instalado y configurado antes de empezar.

In [11]:
# Verificar instalación de dependencias clave
import importlib

deps = ["torch", "torchaudio", "transformers", "pyannote.audio", "soundfile", "yaml", "dotenv"]
for dep in deps:
    try:
        importlib.import_module(dep.replace(".", "_") if dep == "pyannote.audio" else dep)
        print(f"  ✓ {dep}")
    except ImportError:
        print(f"  ✗ {dep} — ejecutar: pip install -r requirements.txt")

  ✓ torch
  ✓ torchaudio
  ✓ transformers
  ✗ pyannote.audio — ejecutar: pip install -r requirements.txt
  ✓ soundfile
  ✓ yaml
  ✓ dotenv


In [12]:
# Verificar ffmpeg
import shutil
if shutil.which("ffmpeg"):
    print("✓ ffmpeg encontrado")
else:
    print("✗ ffmpeg no encontrado — instálalo antes de continuar")

✓ ffmpeg encontrado


In [13]:
# Verificar HF_TOKEN en .env
import os
from src.config import ROOT

env_path = ROOT / ".env"
if env_path.exists():
    print(f"✓ .env encontrado en {env_path}")
else:
    print(f"✗ Falta .env — crear {env_path} con: HF_TOKEN=tu_token_aqui")

token = os.environ.get("HF_TOKEN", "")
if token:
    print(f"✓ HF_TOKEN cargado ({token[:8]}...)")
else:
    print("✗ HF_TOKEN no encontrado en variables de entorno")

✓ .env encontrado en C:\Users\eneko\OneDrive\Belgeler\hawk-and-dove_BCE\.env
✓ HF_TOKEN cargado (hf_FoumT...)


In [14]:
# Configuración común
from src.config import ROOT, RAW_DIR, EVENTS_DIR, load_config, event_dir

cfg = load_config()
fecha = cfg["reference_event"]
ref_dir = event_dir(fecha)
ref_dir.mkdir(parents=True, exist_ok=True)

print(f"Evento de referencia: {fecha}")
print(f"Directorio raw:       {RAW_DIR}")
print(f"Directorio evento:    {ref_dir}")

Evento de referencia: 2026-09-10
Directorio raw:       C:\Users\eneko\OneDrive\Belgeler\hawk-and-dove_BCE\data\raw
Directorio evento:    C:\Users\eneko\OneDrive\Belgeler\hawk-and-dove_BCE\data\events\2026-09-10


## 1. Descargar vídeo y transcripción oficial

Descarga el vídeo de YouTube (vía `yt-dlp`) y la transcripción oficial del BCE.

In [15]:
# Descargar vídeo de la rueda de prensa
import subprocess
video_path = RAW_DIR / f"{fecha}.mp4"

if video_path.exists():
    print(f"✓ Vídeo ya existe: {video_path}")
else:
    print(f"Descargando vídeo de la rueda {fecha}...")
    subprocess.run(["python", str(ROOT / "scripts" / "download_ecb.py"), fecha], check=True)

✓ Vídeo ya existe: C:\Users\eneko\OneDrive\Belgeler\hawk-and-dove_BCE\data\raw\2026-09-10.mp4


In [16]:
# Descargar transcripción oficial
transcript_ref_path = RAW_DIR / f"{fecha}_transcript.txt"

if transcript_ref_path.exists():
    print(f"✓ Transcripción ya existe: {transcript_ref_path}")
else:
    print("⚠ La transcripción se descarga junto con el vídeo en el paso anterior.")
    print("  Si falló, descárgala manualmente de: https://www.ecb.europa.eu/press/pressconf/")

⚠ La transcripción se descarga junto con el vídeo en el paso anterior.
  Si falló, descárgala manualmente de: https://www.ecb.europa.eu/press/pressconf/


## 2. Extraer audio (WAV)

Convierte el vídeo MP4 a WAV mono 16kHz, que es el formato que esperan todos los modelos.

In [17]:
from src.audio.pipeline import extraer_audio

audio_path = RAW_DIR / f"{fecha}.wav"
if audio_path.exists():
    print(f"✓ Audio ya existe: {audio_path}")
else:
    print("Extrayendo audio del vídeo...")
    audio_path = extraer_audio(video_path)
    print(f"✓ Audio extraído: {audio_path}")

# Verificar duración
import soundfile as sf
info = sf.info(str(audio_path))
print(f"  Duración: {info.duration / 60:.1f} minutos")
print(f"  Sample rate: {info.samplerate} Hz")
print(f"  Canales: {info.channels}")

✓ Audio ya existe: C:\Users\eneko\OneDrive\Belgeler\hawk-and-dove_BCE\data\raw\2026-09-10.wav
  Duración: 50.9 minutos
  Sample rate: 16000 Hz
  Canales: 1


## 3. Transcripción (ASR)

Modelo: **Whisper large-v3-turbo** — transcribe el audio completo con timestamps por segmento.

In [9]:
from src.audio.transcription import transcribir

print(f"Transcribiendo {audio_path.name} con Whisper large-v3-turbo...")
print("(esto puede tardar varios minutos dependiendo de la GPU)")
segmentos = transcribir(audio_path)
print(f"\n✓ Segmentos transcritos: {len(segmentos)}")
print("\nPrimeros 5 segmentos:")
print(segmentos.head())

Transcribiendo 2026-09-10.wav con Whisper large-v3-turbo...
(esto puede tardar varios minutos dependiendo de la GPU)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 587/587 [00:01<00:00, 414.24it/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] Passing `generation_config` together with generation-related arguments=({'return_timestamps'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


KeyboardInterrupt: 

In [19]:
import pandas as pd
from src.config import ROOT, RAW_DIR, EVENTS_DIR, load_config, event_dir

cfg = load_config()
fecha = cfg["reference_event"]
ref_dir = event_dir(fecha)
audio_path = RAW_DIR / f"{fecha}.wav"

print(f"Evento: {fecha}")
print(f"Audio: {audio_path}")

segmentos = pd.read_csv(ref_dir / "transcript_raw.csv")
print(f"✓ Segmentos cargados desde transcript_raw.csv: {len(segmentos)}")
segmentos.head()

Evento: 2026-09-10
Audio: C:\Users\eneko\OneDrive\Belgeler\hawk-and-dove_BCE\data\raw\2026-09-10.wav
✓ Segmentos cargados desde transcript_raw.csv: 541


,start,end,text
0,0.00,3.54,"Good afternoon, a warm welcome to all of you."
1,4.42,7.16,It is an honor and a pleasure for the Bundesbank
2,7.16,11.88,to host this external meeting of the Governing...
3,13.16,18.74,Berlin was shaped by the division of Germany a...
4,19.96,25.84,"Today, unity is becoming increasingly importan..."


## 4. Diarización de hablantes

Modelo: **pyannote 3.1** — identifica quién habla en cada momento (presidenta, vicepresidente, periodistas).

⚠ Se carga el audio con `soundfile` en lugar de `torchaudio` para evitar la dependencia de `torchcodec` (incompatible con Windows sin ffmpeg full-shared).
ℹ La salida de pyannote 4.x es un `DiarizeOutput` — los turnos se extraen de `diarization.speaker_diarization`.

In [22]:
import os
import soundfile as sf
import torch
from pyannote.audio import Pipeline

# Cargar audio con soundfile (sin torchcodec)
data, sample_rate = sf.read(str(audio_path))
waveform = torch.from_numpy(data).float()
if waveform.ndim == 1:
    waveform = waveform.unsqueeze(0)
else:
    waveform = waveform.T  # (samples, channels) → (channels, samples)

print(f"Audio cargado: {waveform.shape}, {sample_rate} Hz")

# Cargar pipeline de diarización
hf_token = os.environ.get("HF_TOKEN")
print("Cargando modelo de diarización...")
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", token=hf_token)

# Diarizar
print(f"Diarizando {audio_path.name}...")
diarization = pipeline({"waveform": waveform, "sample_rate": sample_rate})

# Construir DataFrame
turnos = []
for turn, _, speaker in diarization.itertracks(yield_label=True):
    turnos.append({
        "start": round(turn.start, 2),
        "end": round(turn.end, 2),
        "speaker_id": speaker,
    })

diarizacion = pd.DataFrame(turnos, columns=["start", "end", "speaker_id"])
print(f"\n✓ Turnos detectados: {len(diarizacion)}")
print(f"  Hablantes únicos: {diarizacion['speaker_id'].nunique()}")
diarizacion.head(10)

Audio cargado: torch.Size([1, 48828407]), 16000 Hz
Cargando modelo de diarización...
Diarizando 2026-09-10.wav...


c:\Users\eneko\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1879.)
  std = sequences.std(dim=-1, correction=1)


AttributeError: 'DiarizeOutput' object has no attribute 'itertracks'

In [23]:
# Inspeccionar qué devolvió pyannote 4.x
print(type(diarization))
print([a for a in dir(diarization) if not a.startswith('_')])

<class 'pyannote.audio.pipelines.speaker_diarization.DiarizeOutput'>
['exclusive_speaker_diarization', 'serialize', 'speaker_diarization', 'speaker_embeddings']


In [24]:
# Extraer turnos según la versión de pyannote
if hasattr(diarization, 'itertracks'):
    annotation = diarization
elif hasattr(diarization, 'output'):
    annotation = diarization.output
elif hasattr(diarization, 'annotation'):
    annotation = diarization.annotation
else:
    # Si nada funciona, convertir a dict/list
    annotation = diarization[0] if isinstance(diarization, tuple) else diarization

turnos = []
for turn, _, speaker in annotation.itertracks(yield_label=True):
    turnos.append({
        "start": round(turn.start, 2),
        "end": round(turn.end, 2),
        "speaker_id": speaker,
    })

diarizacion = pd.DataFrame(turnos, columns=["start", "end", "speaker_id"])
print(f"✓ Turnos detectados: {len(diarizacion)}")
print(f"  Hablantes únicos: {diarizacion['speaker_id'].nunique()}")
diarizacion.head(10)

AttributeError: 'DiarizeOutput' object has no attribute 'itertracks'

In [26]:
annotation = diarization.speaker_diarization

turnos = []
for turn, _, speaker in annotation.itertracks(yield_label=True):
    turnos.append({
        "start": round(turn.start, 2),
        "end": round(turn.end, 2),
        "speaker_id": speaker,
    })

diarizacion = pd.DataFrame(turnos, columns=["start", "end", "speaker_id"])
print(f"✓ Turnos detectados: {len(diarizacion)}")
print(f"  Hablantes únicos: {diarizacion['speaker_id'].nunique()}")
diarizacion.head(10)

✓ Turnos detectados: 896
  Hablantes únicos: 12


,start,end,speaker_id
0,0.15,2.51,SPEAKER_08
1,2.92,3.66,SPEAKER_08
2,4.50,7.29,SPEAKER_08
3,8.13,9.99,SPEAKER_08
4,10.44,12.06,SPEAKER_08
5,13.19,15.29,SPEAKER_08
6,15.47,16.33,SPEAKER_08
7,17.01,18.95,SPEAKER_08
8,19.94,20.60,SPEAKER_08
9,21.29,24.33,SPEAKER_08


In [27]:
diarizacion.to_csv(ref_dir / "diarization_raw.csv", index=False)
print(f"✓ Guardado en {ref_dir / 'diarization_raw.csv'}")

✓ Guardado en C:\Users\eneko\OneDrive\Belgeler\hawk-and-dove_BCE\data\events\2026-09-10\diarization_raw.csv


## 5. Cruce ASR + Diarización → `transcript.csv`

Asigna a cada segmento transcrito el hablante correspondiente según la diarización.

In [28]:
from src.audio.diarization import asignar_hablantes
import pandas as pd

transcript = asignar_hablantes(segmentos, diarizacion)
transcript_path = ref_dir / "transcript.csv"
transcript.to_csv(transcript_path, index=False)

print(f"✓ transcript.csv guardado: {transcript_path}")
print(f"  Filas: {len(transcript)}")
print(f"  Columnas: {list(transcript.columns)}")
print("\nPrimeras filas:")
print(transcript.head(10))

✓ transcript.csv guardado: C:\Users\eneko\OneDrive\Belgeler\hawk-and-dove_BCE\data\events\2026-09-10\transcript.csv
  Filas: 541
  Columnas: ['start', 'end', 'speaker', 'section', 'text']

Primeras filas:
   start    end     speaker section  \
0   0.00   3.54  periodista      qa   
1   4.42   7.16  periodista      qa   
2   7.16  11.88  periodista      qa   
3  13.16  18.74  periodista      qa   
4  19.96  25.84  periodista      qa   
5  26.78  25.84  periodista      qa   
6  35.52  36.70  periodista      qa   
7  42.76  44.10  periodista      qa   
8  52.28  53.22  periodista      qa   
9  56.70  57.10  periodista      qa   

                                                text  
0      Good afternoon, a warm welcome to all of you.  
1   It is an honor and a pleasure for the Bundesbank  
2  to host this external meeting of the Governing...  
3  Berlin was shaped by the division of Germany a...  
4  Today, unity is becoming increasingly importan...  
5                                  

## 6. Emoción de voz → `voice.csv`

Modelo: **wav2vec2-audeering** — extrae arousal (activación) y valence (sentimiento) por ventanas temporales.

In [30]:
import soundfile as sf
import torch
import torchaudio

def _sf_load(filepath, *args, **kwargs):
    data, sample_rate = sf.read(str(filepath))
    waveform = torch.from_numpy(data).float()
    if waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)
    else:
        waveform = waveform.T
    return waveform, sample_rate

torchaudio.load = _sf_load
print("✓ torchaudio.load parcheado con soundfile (sin torchcodec)")

✓ torchaudio.load parcheado con soundfile (sin torchcodec)


In [31]:
from src.audio.voice_emotion import analizar_emocion

print(f"Analizando emoción de voz con wav2vec2-audeering...")
voice = analizar_emocion(audio_path)
voice_path = ref_dir / "voice.csv"
voice.to_csv(voice_path, index=False)

print(f"\n✓ voice.csv guardado: {voice_path}")
print(f"  Ventanas: {len(voice)}")
print(voice.describe())

Analizando emoción de voz con wav2vec2-audeering...


Loading weights: 100%|██████████| 230/230 [00:00<00:00, 554.79it/s]
[transformers] Wav2Vec2ForSequenceClassification LOAD REPORT from: audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim
Key                        | Status     | 
---------------------------+------------+-
classifier.dense.bias      | UNEXPECTED | 
classifier.dense.weight    | UNEXPECTED | 
classifier.out_proj.weight | UNEXPECTED | 
classifier.out_proj.bias   | UNEXPECTED | 
classifier.bias            | MISSING    | 
projector.weight           | MISSING    | 
projector.bias             | MISSING    | 
classifier.weight          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



✓ voice.csv guardado: C:\Users\eneko\OneDrive\Belgeler\hawk-and-dove_BCE\data\events\2026-09-10\voice.csv
  Ventanas: 3052
             start          end      arousal      valence    dominance
count  3052.000000  3052.000000  3052.000000  3052.000000  3052.000000
mean   1525.500000  1527.499528     0.012099     0.002164     0.000257
std     881.180837   881.180020     0.007342     0.003883     0.001286
min       0.000000     2.000000     0.000000     0.000000     0.000000
25%     762.750000   764.750000     0.006780     0.000000     0.000000
50%    1525.500000  1527.500000     0.011757     0.000000     0.000000
75%    2288.250000  2290.250000     0.016645     0.003180     0.000000
max    3051.000000  3051.780000     0.049705     0.037020     0.017314


## 7. Síntesis de voz (TTS) → `briefing.mp3`

Modelo: **edge-tts (en-GB-SoniaNeural)** — genera el audio del briefing a partir de `summary.json`.

⚠ Este paso solo funciona si el Bloque 2 ya generó `summary.json` con el campo `briefing_tts`.
ℹ Se usa edge-tts en lugar de Kokoro-82M por incompatibilidad de Kokoro con Python 3.13.

In [38]:
import edge_tts

communicate = edge_tts.Communicate(texto, "en-GB-SoniaNeural")
await communicate.save(str(briefing_path))
print("✓ Generado")

✓ Generado


In [39]:
if briefing_path.exists():
    print(f"✓ briefing.mp3 ({briefing_path.stat().st_size / 1024:.0f} KB)")
else:
    print("✗ No se generó")

✓ briefing.mp3 (450 KB)


## 8. Verificación final

Comprobar que todos los archivos de salida de B1 se han generado correctamente.

In [40]:
archivos_b1 = {
    "transcript.csv": ref_dir / "transcript.csv",
    "voice.csv": ref_dir / "voice.csv",
    "briefing.mp3": ref_dir / "briefing.mp3",
}

print(f"Archivos de salida B1 en {ref_dir}:\n")
for nombre, ruta in archivos_b1.items():
    if ruta.exists():
        size_kb = ruta.stat().st_size / 1024
        print(f"  ✓ {nombre} ({size_kb:.0f} KB)")
    else:
        print(f"  ✗ {nombre} — no generado")

print("\n--- Pipeline B1 completado ---")

Archivos de salida B1 en C:\Users\eneko\OneDrive\Belgeler\hawk-and-dove_BCE\data\events\2026-09-10:

  ✓ transcript.csv (51 KB)
  ✓ voice.csv (150 KB)
  ✓ briefing.mp3 (450 KB)

--- Pipeline B1 completado ---


## 1 · Setup y monkey-patch de torchaudio

Importaciones generales y parche de `torchaudio.load` para usar `soundfile` como backend.  
Necesario porque `torchcodec` (backend por defecto en PyTorch 2.12) da error de DLL en Windows.

In [1]:
# === CELDA 1: Setup — ejecutar ANTES que nada ===
import os, json, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import torch

# --- Monkey-patch torchaudio.load para evitar torchcodec ---
import torchaudio

def _sf_load(filepath, *args, **kwargs):
    data, sample_rate = sf.read(str(filepath))
    waveform = torch.from_numpy(data).float()
    if waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)
    else:
        waveform = waveform.T
    return waveform, sample_rate

torchaudio.load = _sf_load
print("✓ Monkey-patch torchaudio.load aplicado")

# --- Rutas ---
from dotenv import load_dotenv
load_dotenv()

PROJECT = Path(r"C:\Users\eneko\OneDrive\Belgeler\hawk-and-dove_BCE")
RAW_DIR = PROJECT / "data" / "raw"
EVENTS_DIR = PROJECT / "data" / "events"

# Fechas pendientes (2026-09-10 ya está procesada)
FECHAS = ["2026-02-05", "2026-03-19", "2026-04-30", "2026-06-11", "2026-07-23"]
print(f"✓ Conferencias a procesar: {len(FECHAS)}")
for f in FECHAS:
    video = RAW_DIR / f"{f}.mp4"
    print(f"  {f}: {'✓' if video.exists() else '✗'} {video.name}")

✓ Monkey-patch torchaudio.load aplicado
✓ Conferencias a procesar: 5
  2026-02-05: ✓ 2026-02-05.mp4
  2026-03-19: ✓ 2026-03-19.mp4
  2026-04-30: ✓ 2026-04-30.mp4
  2026-06-11: ✓ 2026-06-11.mp4
  2026-07-23: ✓ 2026-07-23.mp4


## 2 · Carga de modelos (una sola vez)

Se instancian los tres modelos pesados en memoria para reutilizarlos en el bucle:
- **ASR**: `openai/whisper-large-v3-turbo` (transcripción con timestamps)
- **Diarización**: `pyannote/speaker-diarization-3.1` (identificación de hablantes)
- **Emoción de voz**: `audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim` (arousal, valence, dominance)

In [2]:
# === CELDA 2: Cargar modelos (tarda ~1-2 min) ===
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline as hf_pipeline
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification
from pyannote.audio import Pipeline as PyannotePipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device != "cpu" else torch.float32

# 1. ASR: Whisper large-v3-turbo
print("Cargando Whisper...")
asr_model_id = "openai/whisper-large-v3-turbo"
asr_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    asr_model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True
).to(device)
asr_processor = AutoProcessor.from_pretrained(asr_model_id)
asr_pipe = hf_pipeline(
    "automatic-speech-recognition",
    model=asr_model,
    tokenizer=asr_processor.tokenizer,
    feature_extractor=asr_processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
    return_timestamps=True,
)
print("✓ Whisper cargado")

# 2. Diarización: pyannote
print("Cargando pyannote...")
hf_token = os.environ.get("HF_TOKEN")
diar_pipeline = PyannotePipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1", token=hf_token
)
print("✓ pyannote cargado")

# 3. Emoción de voz: wav2vec2 audEERING
print("Cargando wav2vec2 emoción...")
emo_model_id = "audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim"
emo_processor = Wav2Vec2Processor.from_pretrained(emo_model_id)
emo_model = Wav2Vec2ForSequenceClassification.from_pretrained(emo_model_id).to(device)
emo_model.eval()
print("✓ wav2vec2 emoción cargado")

print("\n✓ Todos los modelos cargados")

c:\Users\eneko\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando Whisper...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 587/587 [00:01<00:00, 417.19it/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] Passing `generation_config` together with generation-related arguments=({'return_timestamps'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✓ Whisper cargado
Cargando pyannote...
✓ pyannote cargado
Cargando wav2vec2 emoción...


Loading weights: 100%|██████████| 230/230 [00:00<00:00, 443.93it/s]
[transformers] Wav2Vec2ForSequenceClassification LOAD REPORT from: audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim
Key                        | Status     | 
---------------------------+------------+-
classifier.dense.weight    | UNEXPECTED | 
classifier.dense.bias      | UNEXPECTED | 
classifier.out_proj.weight | UNEXPECTED | 
classifier.out_proj.bias   | UNEXPECTED | 
projector.bias             | MISSING    | 
classifier.bias            | MISSING    | 
projector.weight           | MISSING    | 
classifier.weight          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ wav2vec2 emoción cargado

✓ Todos los modelos cargados


## 3 · Funciones auxiliares del pipeline

Funciones que encapsulan cada etapa del Bloque 1:
- `extraer_audio(mp4) → wav`: extrae la pista de audio del vídeo descargado.
- `transcribir_audio(wav) → DataFrame`: transcripción ASR con timestamps.
- `diarizar_audio(wav) → DataFrame`: segmentos con speaker_id (compatible con pyannote 4.x).
- `cruzar_transcripcion_diarizacion(transcript, diar) → DataFrame`: asigna hablante y sección (statement/qa) a cada frase.
- `analizar_emocion_voz(wav) → DataFrame`: arousal, valence y dominance en ventanas de 2 s.

In [3]:
# === CELDA 3: Funciones del pipeline ===

def extraer_audio(video_path, output_path):
    """Extrae audio WAV 16kHz mono del vídeo."""
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(video_path), "-vn",
         "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1",
         str(output_path)],
        capture_output=True, check=True,
    )
    return output_path


def transcribir_audio(audio_path):
    """ASR con Whisper → DataFrame(start, end, text)."""
    resultado = asr_pipe(
        str(audio_path),
        return_timestamps=True,
        generate_kwargs={"language": "english", "task": "transcribe"},
    )
    segmentos = []
    for chunk in resultado.get("chunks", []):
        ts = chunk.get("timestamp", (None, None))
        segmentos.append({
            "start": ts[0] if ts[0] is not None else 0.0,
            "end": ts[1] if ts[1] is not None else ts[0] or 0.0,
            "text": chunk["text"].strip(),
        })
    return pd.DataFrame(segmentos, columns=["start", "end", "text"])


def diarizar_audio(audio_path):
    """Diarización con pyannote 4.x → DataFrame(start, end, speaker_id)."""
    waveform, sample_rate = sf.read(str(audio_path))
    waveform = torch.from_numpy(waveform).float()
    if waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)
    else:
        waveform = waveform.T
    
    diarization = diar_pipeline({"waveform": waveform, "sample_rate": sample_rate})
    
    # pyannote 4.x: extraer annotation
    annotation = getattr(diarization, 'speaker_diarization', diarization)
    
    turnos = []
    for turn, _, speaker in annotation.itertracks(yield_label=True):
        turnos.append({
            "start": round(turn.start, 2),
            "end": round(turn.end, 2),
            "speaker_id": speaker,
        })
    return pd.DataFrame(turnos, columns=["start", "end", "speaker_id"])


def cruzar_transcripcion_diarizacion(transcripcion, diarizacion):
    """Asigna hablante y sección a cada segmento."""
    speakers = []
    for _, fila in transcripcion.iterrows():
        seg_start, seg_end = fila["start"], fila["end"]
        overlap_start = diarizacion["start"].clip(lower=seg_start)
        overlap_end = diarizacion["end"].clip(upper=seg_end)
        overlap = (overlap_end - overlap_start).clip(lower=0)
        if overlap.sum() == 0:
            speakers.append("UNKNOWN")
        else:
            speakers.append(diarizacion.loc[overlap.idxmax(), "speaker_id"])
    
    transcripcion = transcripcion.copy()
    transcripcion["speaker_id"] = speakers
    
    # Mapear: más tiempo en primera mitad = presidenta, segundo = vicepresidente
    mitad = diarizacion["end"].max() / 2
    pm = diarizacion[diarizacion["start"] < mitad].copy()
    pm["dur"] = pm["end"].clip(upper=mitad) - pm["start"]
    orden = pm.groupby("speaker_id")["dur"].sum().sort_values(ascending=False).index.tolist()
    
    mapa = {}
    if len(orden) >= 1: mapa[orden[0]] = "presidenta"
    if len(orden) >= 2: mapa[orden[1]] = "vicepresidente"
    for s in orden[2:]: mapa[s] = "periodista"
    mapa["UNKNOWN"] = "periodista"
    
    transcripcion["speaker"] = transcripcion["speaker_id"].map(mapa)
    
    # Sección: antes del primer periodista = statement, después = qa
    periodistas = transcripcion[transcripcion["speaker"] == "periodista"]
    if periodistas.empty:
        transcripcion["section"] = "statement"
    else:
        corte = periodistas.iloc[0]["start"]
        transcripcion["section"] = transcripcion["start"].apply(
            lambda t: "statement" if t < corte else "qa"
        )
    
    return transcripcion[["start", "end", "speaker", "section", "text"]]


def analizar_emocion_voz(audio_path, ventana=2.0, solapamiento=0.5):
    """Emoción de voz con wav2vec2 → DataFrame(start, end, arousal, valence, dominance)."""
    waveform, sr = sf.read(str(audio_path))
    waveform = torch.from_numpy(waveform).float()
    if waveform.ndim > 1:
        waveform = waveform.mean(dim=-1)
    
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(sr, 16000)
        waveform = resampler(waveform)
        sr = 16000
    
    duracion = len(waveform) / sr
    paso = ventana * (1 - solapamiento)
    resultados = []
    t = 0.0
    
    while t < duracion:
        t_end = min(t + ventana, duracion)
        ini = int(t * sr)
        fin = int(t_end * sr)
        fragmento = waveform[ini:fin]
        
        if len(fragmento) < sr * 0.5:
            break
        
        inputs = emo_processor(fragmento.numpy(), sampling_rate=sr, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = emo_model(**inputs).logits.squeeze().cpu().numpy()
        
        resultados.append({
            "start": round(t, 2),
            "end": round(t_end, 2),
            "arousal": float(np.clip(logits[0], 0, 1)),
            "valence": float(np.clip(logits[2], 0, 1)),
            "dominance": float(np.clip(logits[1], 0, 1)),
        })
        t += paso
    
    return pd.DataFrame(resultados, columns=["start", "end", "arousal", "valence", "dominance"])


print("✓ Funciones definidas")

✓ Funciones definidas


## 4 · Procesamiento batch de conferencias

Bucle principal que recorre las fechas pendientes y ejecuta el pipeline completo:
1. Extrae audio del vídeo (`data/raw/{fecha}.mp4` → `data/events/{fecha}/audio.wav`)
2. Transcripción ASR → `transcript_raw.csv`
3. Diarización → `diarization_raw.csv`
4. Cruce transcripción + diarización → `transcript.csv` (con speaker y section)
5. Análisis de emoción de voz → `voice.csv`

Salta las fechas que ya tienen `voice.csv` generado.

In [4]:
# === CELDA 4: Procesar las 5 conferencias ===
import time

for fecha in FECHAS:
    print(f"\n{'='*60}")
    print(f"  PROCESANDO: {fecha}")
    print(f"{'='*60}")
    t0 = time.time()
    
    video_path = RAW_DIR / f"{fecha}.mp4"
    carpeta = EVENTS_DIR / fecha
    carpeta.mkdir(parents=True, exist_ok=True)
    
    # 1. Extraer audio
    audio_path = carpeta / "audio.wav"
    print(f"[1/5] Extrayendo audio...")
    extraer_audio(video_path, audio_path)
    print(f"  ✓ audio.wav")
    
    # 2. Transcribir (ASR)
    print(f"[2/5] Transcribiendo (Whisper)...")
    transcript_raw = transcribir_audio(audio_path)
    transcript_raw.to_csv(carpeta / "transcript_raw.csv", index=False)
    print(f"  ✓ transcript_raw.csv ({len(transcript_raw)} segmentos)")
    
    # 3. Diarizar
    print(f"[3/5] Diarizando (pyannote)...")
    diarizacion = diarizar_audio(audio_path)
    diarizacion.to_csv(carpeta / "diarization_raw.csv", index=False)
    n_speakers = diarizacion["speaker_id"].nunique()
    print(f"  ✓ diarization_raw.csv ({len(diarizacion)} turnos, {n_speakers} hablantes)")
    
    # 4. Cruzar → transcript.csv
    print(f"[4/5] Cruzando transcripción + diarización...")
    transcript = cruzar_transcripcion_diarizacion(transcript_raw, diarizacion)
    transcript.to_csv(carpeta / "transcript.csv", index=False)
    print(f"  ✓ transcript.csv ({len(transcript)} filas)")
    
    # 5. Emoción de voz
    print(f"[5/5] Analizando emoción de voz...")
    voice = analizar_emocion_voz(audio_path)
    voice.to_csv(carpeta / "voice.csv", index=False)
    print(f"  ✓ voice.csv ({len(voice)} ventanas)")
    
    elapsed = time.time() - t0
    print(f"\n  ✓ {fecha} completado en {elapsed/60:.1f} minutos")

print(f"\n{'='*60}")
print(f"  TODAS LAS CONFERENCIAS PROCESADAS")
print(f"{'='*60}")


  PROCESANDO: 2026-02-05
[1/5] Extrayendo audio...
  ✓ audio.wav
[2/5] Transcribiendo (Whisper)...


[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  ✓ transcript_raw.csv (432 segmentos)
[3/5] Diarizando (pyannote)...


c:\Users\eneko\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1879.)
  std = sequences.std(dim=-1, correction=1)


  ✓ diarization_raw.csv (542 turnos, 9 hablantes)
[4/5] Cruzando transcripción + diarización...
  ✓ transcript.csv (432 filas)
[5/5] Analizando emoción de voz...
  ✓ voice.csv (2337 ventanas)

  ✓ 2026-02-05 completado en 34.7 minutos

  PROCESANDO: 2026-03-19
[1/5] Extrayendo audio...
  ✓ audio.wav
[2/5] Transcribiendo (Whisper)...
  ✓ transcript_raw.csv (427 segmentos)
[3/5] Diarizando (pyannote)...
  ✓ diarization_raw.csv (499 turnos, 8 hablantes)
[4/5] Cruzando transcripción + diarización...
  ✓ transcript.csv (427 filas)
[5/5] Analizando emoción de voz...
  ✓ voice.csv (2308 ventanas)

  ✓ 2026-03-19 completado en 31.9 minutos

  PROCESANDO: 2026-04-30
[1/5] Extrayendo audio...
  ✓ audio.wav
[2/5] Transcribiendo (Whisper)...


[transformers] Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


  ✓ transcript_raw.csv (609 segmentos)
[3/5] Diarizando (pyannote)...
  ✓ diarization_raw.csv (690 turnos, 12 hablantes)
[4/5] Cruzando transcripción + diarización...
  ✓ transcript.csv (609 filas)
[5/5] Analizando emoción de voz...
  ✓ voice.csv (3165 ventanas)

  ✓ 2026-04-30 completado en 42.6 minutos

  PROCESANDO: 2026-06-11
[1/5] Extrayendo audio...
  ✓ audio.wav
[2/5] Transcribiendo (Whisper)...


[transformers] Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


  ✓ transcript_raw.csv (539 segmentos)
[3/5] Diarizando (pyannote)...
  ✓ diarization_raw.csv (684 turnos, 11 hablantes)
[4/5] Cruzando transcripción + diarización...
  ✓ transcript.csv (539 filas)
[5/5] Analizando emoción de voz...
  ✓ voice.csv (3007 ventanas)

  ✓ 2026-06-11 completado en 39.3 minutos

  PROCESANDO: 2026-07-23
[1/5] Extrayendo audio...
  ✓ audio.wav
[2/5] Transcribiendo (Whisper)...
  ✓ transcript_raw.csv (508 segmentos)
[3/5] Diarizando (pyannote)...
  ✓ diarization_raw.csv (650 turnos, 11 hablantes)
[4/5] Cruzando transcripción + diarización...
  ✓ transcript.csv (508 filas)
[5/5] Analizando emoción de voz...
  ✓ voice.csv (2611 ventanas)

  ✓ 2026-07-23 completado en 34.3 minutos

  TODAS LAS CONFERENCIAS PROCESADAS


In [7]:
# Verificar conectividad
import subprocess
r = subprocess.run(["ping", "-n", "1", "speech.platform.bing.com"], capture_output=True, text=True)
print(r.stdout)


Haciendo ping a ax-0001.ax-msedge.net [2620:1ec:33::10] con 32 bytes de datos:
Respuesta desde 2620:1ec:33::10: tiempo=50ms 

Estad¡sticas de ping para 2620:1ec:33::10:
    Paquetes: enviados = 1, recibidos = 1, perdidos = 0
    (0% perdidos),
Tiempos aproximados de ida y vuelta en milisegundos:
    M¡nimo = 50ms, M ximo = 50ms, Media = 50ms



## 5 · Generación de briefings de audio (TTS)

Genera un resumen hablado (`briefing.mp3`) para cada conferencia usando `edge-tts`
con la voz `en-GB-SoniaNeural`. Se ejecuta como subproceso para evitar conflictos
con el event loop de Jupyter. El texto del briefing se construye a partir de
`summary.json` (generado por el Bloque 2).

In [11]:
# === CELDA 5: Briefings con edge-tts (vía subprocess) ===
import subprocess, json
from pathlib import Path

for fecha in FECHAS:
    carpeta = EVENTS_DIR / fecha
    summary_path = carpeta / "summary.json"
    briefing_path = carpeta / "briefing.mp3"
    
    if not summary_path.exists():
        print(f"[{fecha}] ✗ No hay summary.json, saltando")
        continue
    
    with open(summary_path, encoding="utf-8") as f:
        summary = json.load(f)
    
    texto = summary.get("briefing_tts", summary.get("briefing", ""))
    if not texto:
        print(f"[{fecha}] ✗ No hay texto para briefing")
        continue
    
    # Guardar texto en fichero temporal para evitar problemas con comillas
    txt_tmp = carpeta / "_briefing_text.txt"
    txt_tmp.write_text(texto, encoding="utf-8")
    
    result = subprocess.run(
        ["edge-tts", "--voice", "en-GB-SoniaNeural",
         "--file", str(txt_tmp),
         "--write-media", str(briefing_path)],
        capture_output=True, text=True
    )
    
    txt_tmp.unlink(missing_ok=True)
    
    if briefing_path.exists() and briefing_path.stat().st_size > 0:
        kb = briefing_path.stat().st_size / 1024
        print(f"[{fecha}] ✓ briefing.mp3 ({kb:.0f} KB)")
    else:
        print(f"[{fecha}] ✗ Error: {result.stderr[:200]}")

print("\n✓ Briefings completados")

[2026-02-05] ✓ briefing.mp3 (487 KB)
[2026-03-19] ✓ briefing.mp3 (515 KB)
[2026-04-30] ✓ briefing.mp3 (463 KB)
[2026-06-11] ✓ briefing.mp3 (436 KB)
[2026-07-23] ✓ briefing.mp3 (436 KB)

✓ Briefings completados


In [10]:
!pip uninstall aiodns -y

Found existing installation: aiodns 4.0.0
Uninstalling aiodns-4.0.0:
  Successfully uninstalled aiodns-4.0.0
